# ATE Phase 1 chunk runner


In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [10]:
from pathlib import Path

GATE_MODEL_PATH = "/content/drive/MyDrive/absa_self_train_phase1/ate_gate_teacher_phase1"
ATE_MODEL_PATH = "/content/drive/MyDrive/absa_self_train_phase1/ate_teacher_phase1"

INPUT_CHUNKS_DIR = Path("/content/drive/MyDrive/electronics_cleaned_2_part2/electronics_cleaned2_part2_chunks")
OUTPUT_ROOT = Path("/content/drive/MyDrive/ate_1_electronics_2")

CATEGORY_NAME = "electronics_p2"
RUN_NAME = "ate1_electronics_p2"

FIRST_CHUNK_NUMBER = 91
LAST_CHUNK_NUMBER = 100
MAX_CHUNKS_TO_RUN = None

GATE_THRESHOLD = 0.90
SPAN_THRESHOLD = 0.85
NEGATIVE_GATE_THRESHOLD = 0.10

GATE_BATCH_SIZE = 512
ATE_BATCH_SIZE = 128
MIN_CUDA_BATCH_SIZE = 8
MAX_LENGTH = 192
USE_LENGTH_BUCKETING = True
EMPTY_CACHE_EVERY_N_BATCHES = 0

RESUME = True
FORCE_REPROCESS = False
COMPRESSION = "snappy"

PARTS_DIR = OUTPUT_ROOT / "parts"
HIGH_DIR = PARTS_DIR / "high"
LOW_DIR = PARTS_DIR / "low"
FINAL_DIR = OUTPUT_ROOT / "final"

FINAL_HIGH_PATH = FINAL_DIR / f"{RUN_NAME}_high.parquet"
FINAL_LOW_PATH = FINAL_DIR / f"{RUN_NAME}_low.parquet"
FINAL_ALL_PATH = FINAL_DIR / f"{RUN_NAME}.parquet"

if FIRST_CHUNK_NUMBER is None:
    FIRST_CHUNK_NUMBER = 1
if int(FIRST_CHUNK_NUMBER) < 1:
    raise ValueError("FIRST_CHUNK_NUMBER must be >= 1")
if LAST_CHUNK_NUMBER is not None and int(LAST_CHUNK_NUMBER) < int(FIRST_CHUNK_NUMBER):
    raise ValueError("LAST_CHUNK_NUMBER must be >= FIRST_CHUNK_NUMBER")


print("input:", INPUT_CHUNKS_DIR)
print("output:", OUTPUT_ROOT)
print("chunk numbers:", FIRST_CHUNK_NUMBER, "to", LAST_CHUNK_NUMBER)


input: /content/drive/MyDrive/electronics_cleaned_2_part2/electronics_cleaned2_part2_chunks
output: /content/drive/MyDrive/ate_1_electronics_2
chunk numbers: 91 to 100


In [3]:
import os
import re
import gc
import time
import uuid
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import pyarrow as pa
import pyarrow.parquet as pq

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification

torch.set_grad_enabled(False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))


device: cuda
gpu: Tesla T4


In [4]:
gate_tokenizer = AutoTokenizer.from_pretrained(GATE_MODEL_PATH, use_fast=True)
gate_model = AutoModelForSequenceClassification.from_pretrained(GATE_MODEL_PATH).to(DEVICE)
gate_model.eval()

ate_tokenizer = AutoTokenizer.from_pretrained(ATE_MODEL_PATH, use_fast=True)
ate_model = AutoModelForTokenClassification.from_pretrained(ATE_MODEL_PATH).to(DEVICE)
ate_model.eval()

if USE_FP16:
    gate_model.half()
    ate_model.half()

ID2LABEL = {int(k): v for k, v in ate_model.config.id2label.items()}

print("gate labels:", gate_model.config.id2label)
print("ate labels:", ID2LABEL)

Loading weights:   0%|          | 0/201 [00:03<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

gate labels: {0: 'no_aspect', 1: 'has_aspect'}
ate labels: {0: 'O', 1: 'B-ASP', 2: 'I-ASP'}


In [5]:
TOKEN_RE = re.compile(r"\b\w+(?:'\w+)?\b")

BASE_COLUMNS = ["parent_asin", "sentence_id", "sentence_text", "rating"]
HIGH_COLUMNS = ["parent_asin", "sentence_id", "sentence_text", "rating", "category_name", "gate_confidence", "aspects", "confidences"]
LOW_COLUMNS = ["parent_asin", "sentence_id", "sentence_text", "rating", "gate_confidence"]
ALL_COLUMNS = ["parent_asin", "sentence_id", "sentence_text", "rating", "category_name", "gate_confidence", "aspects", "confidences", "confidence_bucket"]

HIGH_SCHEMA = pa.schema([
    ("parent_asin", pa.string()),
    ("sentence_id", pa.string()),
    ("sentence_text", pa.string()),
    ("rating", pa.float64()),
    ("category_name", pa.string()),
    ("gate_confidence", pa.float64()),
    ("aspects", pa.list_(pa.string())),
    ("confidences", pa.list_(pa.float64())),
])

LOW_SCHEMA = pa.schema([
    ("parent_asin", pa.string()),
    ("sentence_id", pa.string()),
    ("sentence_text", pa.string()),
    ("rating", pa.float64()),
    ("gate_confidence", pa.float64()),
])

ALL_SCHEMA = pa.schema([
    ("parent_asin", pa.string()),
    ("sentence_id", pa.string()),
    ("sentence_text", pa.string()),
    ("rating", pa.float64()),
    ("category_name", pa.string()),
    ("gate_confidence", pa.float64()),
    ("aspects", pa.list_(pa.string())),
    ("confidences", pa.list_(pa.float64())),
    ("confidence_bucket", pa.string()),
])


def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def tokenize_words(text):
    return [m.group(0) for m in TOKEN_RE.finditer(clean_text(text))]


def normalize_span(span):
    span = re.sub(r"\s+", " ", clean_text(span))
    return span.strip(" \t\n\r.,;:!?()[]{}\"'")


def decode_bio_spans(tokens, labels, confidences):
    spans = []
    i = 0
    while i < len(labels):
        if labels[i] == "B-ASP":
            start = i
            i += 1
            while i < len(labels) and labels[i] == "I-ASP":
                i += 1
            end = i - 1
            text = normalize_span(" ".join(tokens[start:end + 1]))
            if text:
                spans.append({
                    "aspect": text,
                    "confidence": float(np.mean(confidences[start:end + 1])),
                    "start_token": int(start),
                    "end_token": int(end),
                })
        else:
            i += 1

    best = {}
    for span in spans:
        key = span["aspect"]
        if key not in best or span["confidence"] > best[key]["confidence"]:
            best[key] = span
    return sorted(best.values(), key=lambda x: (-x["confidence"], x["aspect"]))


def is_cuda_oom(exc):
    msg = str(exc).lower()
    return isinstance(exc, RuntimeError) and DEVICE == "cuda" and ("out of memory" in msg or "cuda error: out of memory" in msg)


def clear_cuda_cache():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def length_sorted_indices(items, length_fn):
    if not USE_LENGTH_BUCKETING:
        return list(range(len(items)))
    return sorted(range(len(items)), key=lambda i: length_fn(items[i]))


def normalize_base_dtypes(df):
    out = df.copy()
    out["parent_asin"] = out["parent_asin"].astype("string")
    out["sentence_id"] = out["sentence_id"].astype("string")
    out["sentence_text"] = out["sentence_text"].map(clean_text).astype("string")
    out["rating"] = pd.to_numeric(out["rating"], errors="coerce").astype("float64")
    return out

In [6]:
def predict_gate_proba(sentences, batch_size=128, max_length=192, desc="gate"):
    n = len(sentences)
    if n == 0:
        return []

    probs_all = np.empty(n, dtype=np.float32)
    order = length_sorted_indices(sentences, lambda x: len(str(x)))
    pos = 0
    cur_bs = int(batch_size)
    n_batches_done = 0

    with tqdm(total=n, desc=desc, unit="sent", leave=False) as pbar:
        while pos < n:
            sub_indices = order[pos:pos + cur_bs]
            batch = [sentences[i] for i in sub_indices]
            try:
                enc = gate_tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
                enc = {k: v.to(DEVICE, non_blocking=True) for k, v in enc.items()}

                with torch.inference_mode():
                    logits = gate_model(**enc).logits
                    probs = torch.softmax(logits.float(), dim=-1)[:, 1].detach().cpu().numpy()

                probs_all[sub_indices] = probs.astype(np.float32)
                pos += len(sub_indices)
                n_batches_done += 1
                pbar.update(len(sub_indices))

                del enc, logits, probs

                if EMPTY_CACHE_EVERY_N_BATCHES and n_batches_done % EMPTY_CACHE_EVERY_N_BATCHES == 0:
                    clear_cuda_cache()

            except RuntimeError as e:
                if is_cuda_oom(e) and cur_bs > MIN_CUDA_BATCH_SIZE:
                    old_bs = cur_bs
                    cur_bs = max(MIN_CUDA_BATCH_SIZE, cur_bs // 2)
                    print(f"[gate oom] batch {old_bs} -> {cur_bs}")
                    clear_cuda_cache()
                    continue
                raise

    return probs_all.astype(float).tolist()


def predict_ate_spans_batch(sentences, batch_size=64, max_length=192, desc="ate"):
    all_results = [[] for _ in sentences]
    if not sentences:
        return all_results

    tokenized_words = [tokenize_words(x) for x in sentences]
    non_empty_indices = [i for i, toks in enumerate(tokenized_words) if len(toks) > 0]
    if USE_LENGTH_BUCKETING:
        non_empty_indices = sorted(non_empty_indices, key=lambda i: len(tokenized_words[i]))

    pos = 0
    cur_bs = int(batch_size)
    n_batches_done = 0

    with tqdm(total=len(non_empty_indices), desc=desc, unit="sent", leave=False) as pbar:
        while pos < len(non_empty_indices):
            sub_indices = non_empty_indices[pos:pos + cur_bs]
            batch_words = [tokenized_words[i] for i in sub_indices]
            try:
                enc = ate_tokenizer(batch_words, is_split_into_words=True, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
                enc_on_device = {k: v.to(DEVICE, non_blocking=True) for k, v in enc.items()}

                with torch.inference_mode():
                    logits = ate_model(**enc_on_device).logits
                    token_probs = torch.softmax(logits.float(), dim=-1)
                    max_probs_t, pred_ids_t = token_probs.max(dim=-1)

                pred_ids = pred_ids_t.detach().cpu().numpy()
                max_probs = max_probs_t.detach().cpu().numpy()

                for local_i, original_i in enumerate(sub_indices):
                    tokens = tokenized_words[original_i]
                    word_ids = enc.word_ids(batch_index=local_i)
                    word_labels = ["O"] * len(tokens)
                    word_confidences = [0.0] * len(tokens)
                    seen_word_ids = set()

                    for token_idx, word_id in enumerate(word_ids):
                        if word_id is None or word_id in seen_word_ids:
                            continue
                        seen_word_ids.add(word_id)
                        if word_id < len(tokens):
                            pred_id = int(pred_ids[local_i, token_idx])
                            word_labels[word_id] = ID2LABEL[pred_id]
                            word_confidences[word_id] = float(max_probs[local_i, token_idx])

                    all_results[original_i] = decode_bio_spans(tokens, word_labels, word_confidences)

                pos += len(sub_indices)
                n_batches_done += 1
                pbar.update(len(sub_indices))

                del enc, enc_on_device, logits, token_probs, max_probs_t, pred_ids_t, pred_ids, max_probs

                if EMPTY_CACHE_EVERY_N_BATCHES and n_batches_done % EMPTY_CACHE_EVERY_N_BATCHES == 0:
                    clear_cuda_cache()

            except RuntimeError as e:
                if is_cuda_oom(e) and cur_bs > MIN_CUDA_BATCH_SIZE:
                    old_bs = cur_bs
                    cur_bs = max(MIN_CUDA_BATCH_SIZE, cur_bs // 2)
                    print(f"[ate oom] batch {old_bs} -> {cur_bs}")
                    clear_cuda_cache()
                    continue
                raise

    return all_results

In [7]:
def natural_key(path):
    text = str(Path(path).name)
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", text)]


def safe_stem(path):
    stem = Path(path).stem
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("_")
    return stem or "chunk"


def discover_input_chunk_files():
    if INPUT_CHUNKS_DIR.is_file() and INPUT_CHUNKS_DIR.suffix.lower() == ".parquet":
        files = [INPUT_CHUNKS_DIR]
    else:
        files = sorted(INPUT_CHUNKS_DIR.glob("*.parquet"), key=natural_key)
    if not files:
        raise FileNotFoundError(f"no parquet chunks in {INPUT_CHUNKS_DIR}")
    return files


def select_chunk_files(files):
    start = int(FIRST_CHUNK_NUMBER) - 1
    end = len(files) if LAST_CHUNK_NUMBER is None else int(LAST_CHUNK_NUMBER)
    selected = list(enumerate(files, start=1))[start:end]
    if MAX_CHUNKS_TO_RUN is not None:
        selected = selected[:int(MAX_CHUNKS_TO_RUN)]
    return selected


def chunk_id_from_path(file_path):
    return safe_stem(file_path)


def paths_for_chunk(chunk_id):
    return {
        "high": HIGH_DIR / f"{chunk_id}.parquet",
        "low": LOW_DIR / f"{chunk_id}.parquet",
    }


def outputs_done(paths):
    return Path(paths["high"]).exists() and Path(paths["low"]).exists()


def validate_chunk_schema(file_path):
    schema = pq.read_schema(file_path)
    missing = [c for c in BASE_COLUMNS if c not in set(schema.names)]
    if missing:
        raise ValueError(f"{file_path} missing columns: {missing}")


def read_input_chunk(file_path):
    validate_chunk_schema(file_path)
    return pd.read_parquet(file_path, columns=BASE_COLUMNS, engine="pyarrow")


def write_parquet_atomic(df, out_path, schema):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_name(f".{out_path.name}.tmp.{uuid.uuid4().hex}")
    df = df.copy()
    for name in schema.names:
        if name not in df.columns:
            df[name] = pd.Series(dtype="object")
    df = df[schema.names]
    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    pq.write_table(table, tmp, compression=COMPRESSION)
    tmp.replace(out_path)
    return str(out_path)


In [8]:
def process_one_chunk(raw_df, chunk_id):
    df = raw_df[[c for c in BASE_COLUMNS if c in raw_df.columns]].copy()
    missing = [c for c in BASE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"missing columns: {missing}")

    df = normalize_base_dtypes(df)
    sentences = df["sentence_text"].fillna("").astype(str).tolist()

    gate_probs = predict_gate_proba(sentences, batch_size=GATE_BATCH_SIZE, max_length=MAX_LENGTH, desc=f"{chunk_id} gate")
    gate_probs_np = np.asarray(gate_probs, dtype=np.float64)
    df["gate_confidence"] = gate_probs_np

    aspects_all = [[] for _ in range(len(df))]
    confidences_all = [[] for _ in range(len(df))]

    candidate_indices = np.where(gate_probs_np >= GATE_THRESHOLD)[0].tolist()
    if candidate_indices:
        candidate_sentences = [sentences[i] for i in candidate_indices]
        ate_spans = predict_ate_spans_batch(candidate_sentences, batch_size=ATE_BATCH_SIZE, max_length=MAX_LENGTH, desc=f"{chunk_id} ate")
        for row_idx, spans in zip(candidate_indices, ate_spans):
            kept = [s for s in spans if float(s["confidence"]) >= SPAN_THRESHOLD]
            aspects_all[row_idx] = [str(s["aspect"]) for s in kept]
            confidences_all[row_idx] = [float(s["confidence"]) for s in kept]

    has_kept_aspect = np.asarray([len(x) > 0 for x in aspects_all], dtype=bool)
    high_mask = (gate_probs_np <= NEGATIVE_GATE_THRESHOLD) | ((gate_probs_np >= GATE_THRESHOLD) & has_kept_aspect)

    high_indices = np.where(high_mask)[0].tolist()
    low_indices = np.where(~high_mask)[0].tolist()

    high_df = df.iloc[high_indices][BASE_COLUMNS + ["gate_confidence"]].copy()
    high_df["category_name"] = CATEGORY_NAME
    high_df["aspects"] = [aspects_all[i] for i in high_indices]
    high_df["confidences"] = [confidences_all[i] for i in high_indices]
    high_df = high_df[HIGH_COLUMNS]

    low_df = df.iloc[low_indices][LOW_COLUMNS].copy()

    high_df = normalize_base_dtypes(high_df)
    high_df["category_name"] = high_df["category_name"].astype("string")
    high_df["gate_confidence"] = high_df["gate_confidence"].astype("float64")
    high_df["aspects"] = high_df["aspects"].map(lambda x: list(x) if isinstance(x, list) else [])
    high_df["confidences"] = high_df["confidences"].map(lambda x: [float(v) for v in x] if isinstance(x, list) else [])

    low_df = normalize_base_dtypes(low_df)
    low_df["gate_confidence"] = low_df["gate_confidence"].astype("float64")

    stats = {
        "rows": int(len(df)),
        "high_rows": int(len(high_df)),
        "low_rows": int(len(low_df)),
        "ate_candidate_rows": int(len(candidate_indices)),
        "high_negative_rows": int((high_df["gate_confidence"] <= NEGATIVE_GATE_THRESHOLD).sum()) if len(high_df) else 0,
        "high_aspect_rows": int(high_df["aspects"].map(len).gt(0).sum()) if len(high_df) else 0,
        "total_aspects": int(high_df["aspects"].map(len).sum()) if len(high_df) else 0,
    }
    return high_df[HIGH_COLUMNS], low_df[LOW_COLUMNS], stats

In [11]:
all_files = discover_input_chunk_files()
selected_files = select_chunk_files(all_files)

print(f"all chunks: {len(all_files):,}")
print(f"selected chunks: {len(selected_files):,}")

if selected_files:
    print("first selected:", selected_files[0][0], selected_files[0][1].name)
    print("last selected:", selected_files[-1][0], selected_files[-1][1].name)

counts = {"done": 0, "skipped": 0, "error": 0}

for chunk_number, file_path in tqdm(selected_files, desc="chunks", unit="chunk"):
    chunk_id = f"chunk_{chunk_number}"
    paths = paths_for_chunk(chunk_id)

    if RESUME and not FORCE_REPROCESS and outputs_done(paths):
        counts["skipped"] += 1
        print(f"skip chunk {chunk_number}: {chunk_id}")
        continue

    t0 = time.perf_counter()

    try:
        print(f"processing chunk {chunk_number}: {chunk_id}")

        raw_df = read_input_chunk(file_path)
        print(f"rows: {len(raw_df):,}")

        high_df, low_df, stats = process_one_chunk(raw_df, chunk_id)

        write_parquet_atomic(high_df, paths["high"], HIGH_SCHEMA)
        write_parquet_atomic(low_df, paths["low"], LOW_SCHEMA)

        counts["done"] += 1
        print(
            f"done chunk {chunk_number}: {chunk_id} "
            f"high={stats['high_rows']:,} low={stats['low_rows']:,} "
            f"seconds={time.perf_counter() - t0:.2f}"
        )

        del raw_df, high_df, low_df
        gc.collect()
        clear_cuda_cache()

    except Exception:
        counts["error"] += 1
        raise

print(counts)


all chunks: 100
selected chunks: 10
first selected: 91 reviews_cleaned2_electronics_part2_chunk91.parquet
last selected: 100 reviews_cleaned2_electronics_part2_chunk100.parquet


chunks:   0%|          | 0/10 [00:00<?, ?chunk/s]

processing chunk 91: chunk_91
rows: 26,045


chunk_91 gate:   0%|          | 0/26045 [00:00<?, ?sent/s]

chunk_91 ate:   0%|          | 0/14336 [00:00<?, ?sent/s]

...

chunk_99 ate:   0%|          | 0/16470 [00:00<?, ?sent/s]

done chunk 99: chunk_99 high=19,782 low=7,739 seconds=24.77
processing chunk 100: chunk_100
rows: 27,103


chunk_100 gate:   0%|          | 0/27103 [00:00<?, ?sent/s]

chunk_100 ate:   0%|          | 0/16267 [00:00<?, ?sent/s]

done chunk 100: chunk_100 high=19,572 low=7,531 seconds=24.88
{'done': 10, 'skipped': 0, 'error': 0}


In [ ]:
def part_files(directory):
    return sorted(Path(directory).glob("*.parquet"), key=natural_key)


def write_empty_parquet(out_path, schema):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_name(f".{out_path.name}.tmp.{uuid.uuid4().hex}")
    table = pa.Table.from_pydict({name: [] for name in schema.names}, schema=schema)
    pq.write_table(table, tmp, compression=COMPRESSION)
    tmp.replace(out_path)


def merge_same_schema(files, out_path, schema):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_name(f".{out_path.name}.tmp.{uuid.uuid4().hex}")

    if not files:
        write_empty_parquet(out_path, schema)
        return str(out_path)

    writer = pq.ParquetWriter(tmp, schema, compression=COMPRESSION)
    try:
        for p in tqdm(files, desc=out_path.name, unit="file"):
            table = pq.read_table(p, schema=schema)
            writer.write_table(table)
    finally:
        writer.close()

    tmp.replace(out_path)
    return str(out_path)


def normalize_all_df(df, bucket):
    out = df.copy()
    out["confidence_bucket"] = bucket

    if "category_name" not in out.columns:
        out["category_name"] = CATEGORY_NAME
    if "aspects" not in out.columns:
        out["aspects"] = [[] for _ in range(len(out))]
    if "confidences" not in out.columns:
        out["confidences"] = [[] for _ in range(len(out))]

    out["category_name"] = out["category_name"].fillna(CATEGORY_NAME).astype("string")
    out["aspects"] = out["aspects"].map(lambda x: list(x) if isinstance(x, list) else [])
    out["confidences"] = out["confidences"].map(lambda x: [float(v) for v in x] if isinstance(x, list) else [])
    return out[ALL_COLUMNS]


def merge_all(high_files, low_files, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_name(f".{out_path.name}.tmp.{uuid.uuid4().hex}")

    writer = pq.ParquetWriter(tmp, ALL_SCHEMA, compression=COMPRESSION)
    try:
        for p in tqdm(high_files, desc=f"{out_path.name} high", unit="file"):
            df = pd.read_parquet(p, engine="pyarrow")
            table = pa.Table.from_pandas(normalize_all_df(df, "high"), schema=ALL_SCHEMA, preserve_index=False)
            writer.write_table(table)

        for p in tqdm(low_files, desc=f"{out_path.name} low", unit="file"):
            df = pd.read_parquet(p, engine="pyarrow")
            table = pa.Table.from_pandas(normalize_all_df(df, "low"), schema=ALL_SCHEMA, preserve_index=False)
            writer.write_table(table)
    finally:
        writer.close()

    tmp.replace(out_path)
    return str(out_path)


high_files = part_files(HIGH_DIR)
low_files = part_files(LOW_DIR)

print("high part files:", len(high_files))
print("low part files:", len(low_files))

merge_same_schema(high_files, FINAL_HIGH_PATH, HIGH_SCHEMA)
merge_same_schema(low_files, FINAL_LOW_PATH, LOW_SCHEMA)
merge_all(high_files, low_files, FINAL_ALL_PATH)

print("final high:", FINAL_HIGH_PATH)
print("final low:", FINAL_LOW_PATH)
print("final all:", FINAL_ALL_PATH)


In [ ]:
import pyarrow.parquet as pq
import numpy as np

table_high = pq.read_table(FINAL_HIGH_PATH, columns=['aspects'])
table_low = pq.read_table(FINAL_LOW_PATH, columns=[])

num_high = len(table_high)
num_low = len(table_low)
total_all = num_high + num_low

pass_rate = (num_high / total_all) * 100 if total_all > 0 else 0

aspects_col = table_high.column('aspects').to_pylist()


total_aspects = sum(len(a) for a in aspects_col)
num_has_aspect = sum(1 for a in aspects_col if len(a) > 0)

ratio_has_aspect = (num_has_aspect / num_high) * 100 if num_high > 0 else 0
avg_aspects_per_sample = total_aspects / num_high if num_high > 0 else 0

print(f"--- Thống kê Chi tiết (No-Pandas) ({RUN_NAME}) ---")
print(f"Tổng số bản ghi (High + Low): {total_all:,}")
print(f"Số bản ghi High: {num_high:,}")
print(f"Tỉ lệ sample pass: {pass_rate:.2f}%")
print(f"---")
print(f"Tổng số aspect trong High: {total_aspects:,}")
print(f"Số câu High có aspect: {num_has_aspect:,} ({ratio_has_aspect:.2f}%)")
print(f"Trung bình aspect/sample (High): {avg_aspects_per_sample:.2f}")

del table_high, table_low, aspects_col